<a href="https://colab.research.google.com/github/jeshmin-shrestha/RAG-Document-Q-A-Project/blob/main/RAG_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  RAG Document Question Answering
Ask questions about any PDF, TXT or DOCX file using AI.

**Run every cell top to bottom, then use the interface at the bottom.**


In [1]:
!pip install -q pypdf python-docx sentence-transformers faiss-cpu gradio groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.7 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import faiss
import gradio as gr


from pypdf import PdfReader
from docx import Document as DocxDocument
from sentence_transformers import SentenceTransformer

from groq import Groq
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
print(" Imports ready")

 Imports ready


In [6]:
def extract_text(file_path: str) -> str:
    ext = file_path.lower().rsplit(".", 1)[-1]

    if ext == "pdf":
        text = ""
        try:
            reader = PdfReader(file_path, strict=False)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"
        except Exception:
            pass
        if not text.strip():          # fallback: treat as plain text
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

    elif ext == "docx":
        doc = DocxDocument(file_path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())

    else:                              # .txt or anything else
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

    return text.strip()


def chunk_text(text: str, chunk_size=60, overlap=15) -> list:
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks


print("Loading embedding model...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model ready")


def build_index(chunks):
    vecs = embedder.encode(chunks, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(vecs)
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return idx


def retrieve(query, index, chunks, top_k=5):
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, top_k)
    return [(chunks[i], float(s)) for s, i in zip(scores[0], ids[0])
            if i < len(chunks) and s > 0]


def generate_answer(question: str, context: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",   # fast and free
        messages=[{
            "role": "user",
            "content": f"""Answer the question using ONLY the context below.
If the answer is not in the context, say: "I could not find the answer in the document."

Context:
{context}

Question: {question}"""
        }],
        max_tokens=1024
    )
    return response.choices[0].message.content.strip()


print(" All functions ready")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready
 All functions ready


In [7]:
# Global state
_index  = None
_chunks = []
_name   = ""


def process_file(file):
    global _index, _chunks, _name

    if file is None:
        return " Please upload a file."

    path  = file if isinstance(file, str) else file.name
    _name = path.split("/")[-1]

    try:
        text    = extract_text(path)
        if not text:
            return "No text could be extracted from this file."

        _chunks = chunk_text(text)
        _index  = build_index(_chunks)

        return (
            f" **{_name}** processed!\n\n"
            f"- Characters: {len(text):,}\n"
            f"- Chunks: {len(_chunks)}\n\n"
            f"Now ask your question "
        )
    except Exception as e:
        return f" Error: {e}"


def ask(question):
    if _index is None:
        return " Upload and process a document first."
    if not question.strip():
        return " Please enter a question."

    try:
        results = retrieve(question, _index, _chunks, top_k=5)
        context = "\n\n".join(chunk for chunk, _ in results)
        answer  = generate_answer(question, context)

        snippets = "\n".join(
            f"[{i+1}] (score {s:.2f}) {c[:100]}..."
            for i, (c, s) in enumerate(results)
        )
        return f"{answer}\n\n---\n**Sources used:**\n{snippets}"

    except Exception as e:
        return f" Error: {e}"


print(" RAG pipeline ready")

 RAG pipeline ready


In [10]:
# Terminal-based RAG Q&A
file_path = "/content/Cell Notes.pdf"  # pdf file in directory

# Process the document
text = extract_text(file_path)
chunks = chunk_text(text, chunk_size=60, overlap=15)
index = build_index(chunks)
print(f"Loaded {len(chunks)} chunks from document\n")
print("=" * 50)
print("Ask questions about your document. Type 'exit' to quit.")
print("=" * 50)

while True:
    question = input("\n Your question: ").strip()

    if question.lower() == "exit":
        print("Bye!")
        break

    if not question:
        print("Please enter a question.")
        continue

    results = retrieve(question, index, chunks, top_k=5)
    context = "\n\n".join(chunk for chunk, _ in results)
    answer = generate_answer(question, context)
    print(f"\n Answer: {answer}\n")
    print("-" * 50)

Loaded 8 chunks from document

Ask questions about your document. Type 'exit' to quit.

 Your question: what is this document?

 Answer: This document appears to be study notes for a biology course, specifically Chapter 3: Cell Biology.

--------------------------------------------------

 Your question: what is cell?

 Answer: The cell is the basic unit of life. All living organisms are made up of cells.

--------------------------------------------------

 Your question: what is mitochondria?

 Answer: Mitochondria is known as the powerhouse of the cell, it produces ATP through cellular respiration and has its own DNA.

--------------------------------------------------

 Your question: what is the difference between prokaryotic and eukaryotic  cells?

 Answer: The difference between prokaryotic and eukaryotic cells is:

- Prokaryotic cells: 
  * No membrane-bound nucleus
  * DNA is found in the cytoplasm
  * Smaller in size (1-10 micrometers)
  * Examples: bacteria, archaea

- Eukar

In [12]:
with gr.Blocks(title="RAG Q&A", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    #  RAG Document Q&A
    Upload a PDF, TXT or DOCX file then ask anything about it.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Step 1: Upload")
            upload = gr.File(label="Upload your document", file_types=[".pdf", ".txt", ".docx"])
            proc_btn = gr.Button(" Process Document", variant="primary")
            status = gr.Markdown()

        with gr.Column(scale=2):
            gr.Markdown("### Step 2: Ask")
            question = gr.Textbox(
                label="Your question",
                placeholder="e.g. What is the main topic of this document?",
                lines=3
            )
            ask_btn = gr.Button(" Ask Question", variant="primary")
            answer = gr.Markdown(label="Answer")

    proc_btn.click(fn=process_file, inputs=upload, outputs=status)
    ask_btn.click(fn=ask, inputs=question, outputs=answer)
    question.submit(fn=ask, inputs=question, outputs=answer)

demo.launch(share=True)

/tmp/ipykernel_706/3073847410.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="RAG Q&A", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2446ad40b043f84eaa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
